In [1]:
import torch
from pykeops.torch import LazyTensor

import torch
import gc

def free_gpu_memory():
    # 1. Python Müllabfuhr: Löscht unreferenzierte Objekte
    gc.collect()
    
    # 2. PyTorch Cache leeren: Gibt reservierten Speicher an die GPU zurück
    torch.cuda.empty_cache()
    
    # (Optional) Sicherstellen, dass alles erledigt ist
    torch.cuda.synchronize()
    
    print("GPU Speicher bereinigt.")

# Aufruf
free_gpu_memory()


import gc

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    if device == 'cuda':
        torch.cuda.synchronize()

GPU Speicher bereinigt.


In [8]:
import torch
import time
import math
import gc
torch.set_float32_matmul_precision('high')
# Versuche Triton zu importieren
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False
    print("WARNUNG: Triton konnte nicht importiert werden.")

# Versuche KeOps zu importieren
try:
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
except ImportError:
    HAS_KEOPS = False

# --- KONFIGURATION ---
N = 2**17        # 131.072 Teilchen
D = 3             
CHUNK = 2**13    # 8192 (Standard)
r0 = .1
c = 1.0
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"--- BENCHMARK ---")
print(f"N={N}, Device={device}")

# ============================================================================
# 1. TRITON KERNEL (Safe)
# ============================================================================
if HAS_TRITON:
    @triton.jit
    def triton_kernel_DxN_safe(
        ptr_q, ptr_forces,
        N: tl.int32, sigma_sq: tl.float32, V0: tl.float32,
        BLOCK_SIZE: tl.constexpr
    ):
        pid = tl.program_id(axis=0)
        #if pid >= N: return

        prefactor = V0 / sigma_sq
        inv_width = -1.0 / (2.0 * sigma_sq)
        
        off_x = pid; off_y = N + pid; off_z = 2 * N + pid
        q_i_x = tl.load(ptr_q + off_x)
        q_i_y = tl.load(ptr_q + off_y)
        q_i_z = tl.load(ptr_q + off_z)
        
        acc_fx = 0.0; acc_fy = 0.0; acc_fz = 0.0
        
        for j_start in range(0, N, BLOCK_SIZE):
            offs = tl.arange(0, BLOCK_SIZE)
            j_idxs = j_start + offs
            mask = j_idxs < N
            
            q_j_x = tl.load(ptr_q + j_idxs, mask=mask, other=0.0)
            q_j_y = tl.load(ptr_q + N + j_idxs, mask=mask, other=0.0)
            q_j_z = tl.load(ptr_q + 2 * N + j_idxs, mask=mask, other=0.0)
            
            dx = q_i_x - q_j_x
            dy = q_i_y - q_j_y
            dz = q_i_z - q_j_z
            
            r_sq = dx*dx + dy*dy + dz*dz
            r_sq = tl.where(mask, r_sq, float('inf')) # Masking
            
            val = tl.exp(inv_width * r_sq)
            force_mag = val * prefactor
            
            acc_fx += tl.sum(dx * force_mag)
            acc_fy += tl.sum(dy * force_mag)
            acc_fz += tl.sum(dz * force_mag)

        tl.store(ptr_forces + off_x, acc_fx)
        tl.store(ptr_forces + off_y, acc_fy)
        tl.store(ptr_forces + off_z, acc_fz)

    def run_triton_safe(q, sigma, V0, chunk=None): 
        if not q.is_contiguous(): q = q.contiguous()
        D, N = q.shape
        forces = torch.empty_like(q)
        grid = (N,) 
        triton_kernel_DxN_safe[grid](q, forces, N, sigma**2, V0, BLOCK_SIZE=2048, num_warps=4)
        return forces, 0

# ============================================================================
# 2. PYTORCH MATMUL (NEU: Schnell & Speichereffizient!)
# ============================================================================
@torch.compile(mode="reduce-overhead")
def pytorch_matmul_DxN(q, sigma, V0, chunk=8192):
    """
    Nutzt r^2 = a^2 + b^2 - 2ab via Matrix-Multiplikation.
    Vermeidet den 3D-Tensor und ist viel schneller als 'Chunked'.
    """
    D, N = q.shape
    sigma_sq = sigma**2
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    
    # |q|^2 vorausberechnen (N,)
    q_sq = (q**2).sum(dim=0)
    
    forces = torch.empty_like(q)

    for i in range(0, N, chunk):
        end = min(i + chunk, N)
        
        # Slices
        q_chunk = q[:, i:end]       # (D, Chunk)
        q_sq_chunk = q_sq[i:end]    # (Chunk,)
        
        # 1. ABSTÄNDE via MatMul (Nutzt Tensor Cores!)
        # (Chunk, D) @ (D, N) -> (Chunk, N)
        term_dot = torch.matmul(q_chunk.T, q)
        
        # r^2 = a^2 + b^2 - 2ab
        r_sq = q_sq_chunk[:, None] + q_sq[None, :] - 2 * term_dot
        r_sq = torch.clamp(r_sq, min=0.0) # Numerische Stabilität
        
        # 2. KRAFT
        exp_term = torch.exp(inv_width * r_sq)
        force_magnitude = exp_term * prefactor_force
        
        # F_i = q_i * sum(W) - sum(q_j * W)
        sum_weights = force_magnitude.sum(dim=1)     
        term_push = q_chunk * sum_weights[None, :]   
        
        # (D, N) @ (N, Chunk) -> (D, Chunk)
        term_pull = torch.matmul(q, force_magnitude.T)
        
        forces[:, i:end] = term_push - term_pull

    return forces, 0

# ============================================================================
# 3. PYTORCH BROADCASTING (Alt & Langsam)
# ============================================================================
#@torch.compile(mode="reduce-overhead")
def chunked_DxN(q, sigma, V0, chunk=CHUNK):
    # WARNUNG: Braucht extrem viel Speicher (D, Chunk, N)
    D, N = q.shape
    sigma_sq = sigma**2
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    forces = torch.empty_like(q)

    for i in range(0, N, chunk):
        q_chunk = q[:, i:i+chunk] 
        diff = q_chunk[:, :, None] - q[:, None, :] 
        r_sq = (diff**2).sum(dim=0)
        exp_term = torch.exp(inv_width * r_sq)
        force_magnitude = exp_term * prefactor_force
        force_vecs = diff * force_magnitude[None, :, :]
        forces[:, i:i+chunk] = force_vecs.sum(dim=2)
    return forces, 0

# ============================================================================
# 4. KEOPS
# ============================================================================
def keops_DxN(positions, r0, c, chunk=None):
    pos_NxD = positions.T.contiguous() 
    N = pos_NxD.shape[0]; r0_2 = r0**2
    x_i = LazyTensor(pos_NxD[:, None, :])
    x_j = LazyTensor(pos_NxD[None, :, :])
    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term 
    forces_NxD = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces_NxD.T, 0

# ============================================================================
# MAIN
# ============================================================================
q_NxD = torch.randn(N, D, device=device, dtype=dtype)
q_DxN = q_NxD.T.contiguous() 

algos = []

# PyTorch
algos.append(("Chunked (DxN)", chunked_DxN,      q_DxN, "DxN"))
algos.append(("MatMul  (DxN)", pytorch_matmul_DxN, q_DxN, "DxN")) # <-- HIER EINGEFÜGT

if HAS_TRITON:
    algos.append(("Triton Safe(DxN)", run_triton_safe, q_DxN, "DxN"))

if HAS_KEOPS:
    algos.append(("KeOps     (DxN)", keops_DxN,       q_DxN, "DxN"))

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35}")
print("-" * 90)

# 1. WARMUP
print("Warmup läuft...")
for name, func, data, layout in algos:
    try:
        if "Chunked" in name: c_size = 512
        elif "MatMul" in name: c_size = 4096
        else: c_size = None
        func(data, r0, c, chunk=c_size)
    except Exception as e:
        print(f"Warmup Fehler bei {name}: {e}")
print("Warmup fertig.\n")

# 2. RUN
for name, func, data, layout in algos:
    gc.collect(); torch.cuda.empty_cache(); 
    if device == 'cuda': torch.cuda.synchronize()

    start = time.perf_counter()
    
    # --- INTELLIGENTE CHUNK WAHL ---
    if "Chunked" in name:
        # Broadcasting frisst RAM -> Kleiner Chunk
        current_chunk = 1024
    elif "MatMul" in name:
        # MatMul ist effizient -> Großer Chunk für Tensor Cores!
        current_chunk = 8192*2
    else:
        current_chunk = None

    f, p = func(data, r0, c, chunk=current_chunk)

    if device == 'cuda': torch.cuda.synchronize()
    duration = time.perf_counter() - start

    f0 = f[:, 0] if layout == "DxN" else f[0, :]
    f0_str = f"[{f0[0]:.5f}, {f0[1]:.5f}, {f0[2]:.5f}]"
    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35}")

print("-" * 90)

--- BENCHMARK ---
N=131072, Device=cuda
ALGORITHMUS          | ZEIT (s) | KRAFT TEILCHEN 0                   
------------------------------------------------------------------------------------------
Warmup läuft...
Warmup fertig.

Chunked (DxN)        | 3.08883  | [6.82037, -97.31467, -7.30381]     
MatMul  (DxN)        | 0.94399  | [6.62789, -97.60314, -7.25492]     
Triton Safe(DxN)     | 0.04073  | [6.82038, -97.31464, -7.30380]     
KeOps     (DxN)      | 0.02412  | [6.82037, -97.31465, -7.30381]     
------------------------------------------------------------------------------------------


In [3]:
import torch
import time
import math
import gc
asd Versuche Triton zu importieren
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False
    print("WARNUNG: Triton konnte nicht importiert werden.")

# Versuche KeOps zu importieren
try:
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
except ImportError:
    HAS_KEOPS = False

# --- KONFIGURATION ---
N = 2**17        # 16.384 Teilchen (Für Tests). Bei A6000 kannst du höher gehen (2**17)
D = 3             
CHUNK = 2**13     # Standard Chunk
r0 = .1
c = 1.0
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"--- BENCHMARK ---")
print(f"N={N}, Device={device}")

# ============================================================================
# 1. TRITON KERNEL (Safe - mit Masken für krumme N)
# ============================================================================
if HAS_TRITON:
    @triton.jit
    def triton_kernel_DxN_safe(
        ptr_q, ptr_forces,
        N: tl.int32, sigma_sq: tl.float32, V0: tl.float32,
        BLOCK_SIZE: tl.constexpr
    ):
        pid = tl.program_id(axis=0)
        
        # Sicherheitscheck: Falls Grid größer als N ist
        if pid >= N:
            return

        prefactor = V0 / sigma_sq
        inv_width = -1.0 / (2.0 * sigma_sq)
        
        off_x = pid
        off_y = N + pid
        off_z = 2 * N + pid
        
        q_i_x = tl.load(ptr_q + off_x)
        q_i_y = tl.load(ptr_q + off_y)
        q_i_z = tl.load(ptr_q + off_z)
        
        acc_fx = 0.0; acc_fy = 0.0; acc_fz = 0.0
        
        # Loop mit Masken-Check für den letzten Block
        for j_start in range(0, N, BLOCK_SIZE):
            offs = tl.arange(0, BLOCK_SIZE)
            j_idxs = j_start + offs
            
            # Maske: True für gültige Teilchen
            mask = j_idxs < N
            
            # Safe Load (lädt 0.0 bei Überhang)
            q_j_x = tl.load(ptr_q + j_idxs, mask=mask, other=0.0)
            q_j_y = tl.load(ptr_q + N + j_idxs, mask=mask, other=0.0)
            q_j_z = tl.load(ptr_q + 2 * N + j_idxs, mask=mask, other=0.0)
            
            dx = q_i_x - q_j_x
            dy = q_i_y - q_j_y
            dz = q_i_z - q_j_z
            
            r_sq = dx*dx + dy*dy + dz*dz
            
            # Maskiere ungültige Teilchen (r_sq = inf -> exp(-inf) = 0)
            r_sq = tl.where(mask, r_sq, float('inf'))
            
            val = tl.exp(inv_width * r_sq)
            force_mag = val * prefactor
            
            acc_fx += tl.sum(dx * force_mag)
            acc_fy += tl.sum(dy * force_mag)
            acc_fz += tl.sum(dz * force_mag)

        tl.store(ptr_forces + off_x, acc_fx)
        tl.store(ptr_forces + off_y, acc_fy)
        tl.store(ptr_forces + off_z, acc_fz)

    def run_triton_safe(q, sigma, V0, chunk=None): # chunk Argument für Kompatibilität ignoriert
        if not q.is_contiguous(): q = q.contiguous()
        D, N = q.shape
        forces = torch.empty_like(q)
        grid = (N,)
        triton_kernel_DxN_safe[grid](q, forces, N, sigma**2, V0, BLOCK_SIZE=2048, num_warps=4)
        return forces, 0

# ============================================================================
# 2. PYTORCH BROADCASTING (Speicherintensiv!)
# ============================================================================
@torch.compile()
def chunked_DxN(q, sigma, V0, chunk=CHUNK):
    D, N = q.shape
    sigma_sq = sigma**2
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    
    forces = torch.empty_like(q)

    # PyTorch Slicing (i:i+chunk) handled krumme N automatisch (der letzte Slice ist kürzer)
    for i in range(0, N, chunk):
        q_chunk = q[:, i:i+chunk] 

        # 3D Tensor Explosion hier:
        diff = q_chunk[:, :, None] - q[:, None, :] 

        r_sq = (diff**2).sum(dim=0)
        exp_term = torch.exp(inv_width * r_sq)
        force_magnitude = exp_term * prefactor_force
        force_vecs = diff * force_magnitude[None, :, :]

        forces[:, i:i+chunk] = force_vecs.sum(dim=2)

    return forces, 0

 

# ============================================================================
# 3. KEOPS
# ============================================================================
def keops_DxN(positions, r0, c, chunk=None):
    pos_NxD = positions.T.contiguous() 
    N = pos_NxD.shape[0]; r0_2 = r0**2

    x_i = LazyTensor(pos_NxD[:, None, :])
    x_j = LazyTensor(pos_NxD[None, :, :])

    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term 

    forces_NxD = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces_NxD.T, 0

# ============================================================================
# MAIN
# ============================================================================
q_NxD = torch.randn(N, D, device=device, dtype=dtype)
q_DxN = q_NxD.T.contiguous() # (3, N) layout

algos = []

# PyTorch Broadcasting Methoden
algos.append(("Chunked (DxN)", chunked_DxN, q_DxN, "DxN"))
 
# Triton
if HAS_TRITON:
    algos.append(("Triton Safe(DxN)", run_triton_safe, q_DxN, "DxN"))

# KeOps
if HAS_KEOPS:
    algos.append(("KeOps     (DxN)", keops_DxN,       q_DxN, "DxN"))

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35}")
print("-" * 90)

# 1. WARMUP
print("Warmup läuft...")
for name, func, data, layout in algos:
    try:
        # Kleiner Chunk für Warmup der Broadcasting-Methoden
        c_size = 512 if "Chunked" in name else CHUNK
        func(data, r0, c, chunk=c_size)
    except Exception as e:
        print(f"Warmup Fehler bei {name}: {e}")
print("Warmup fertig.\n")

# 2. RUN
for name, func, data, layout in algos:
    # Cleanup (Extrem wichtig für Broadcasting!)
    gc.collect()
    torch.cuda.empty_cache()
    if device == 'cuda': torch.cuda.synchronize()

    start = time.perf_counter()
    
    # Intelligente Chunk-Wahl
    # Broadcasting braucht viel RAM -> Kleiner Chunk
    # Triton/KeOps brauchen keinen Chunk Parameter
    if "Chunked" in name:
        f, p = func(data, r0, c, chunk=1024) 
    else:
        f, p = func(data, r0, c, chunk=None)

    if device == 'cuda': torch.cuda.synchronize()
    duration = time.perf_counter() - start

    if layout == "DxN": f0 = f[:, 0]
    else: f0 = f[0, :]
    
    f0_str = f"[{f0[0]:.5f}, {f0[1]:.5f}, {f0[2]:.5f}]"
    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35}")

print("-" * 90)

SyntaxError: invalid syntax (845456.py, line 5)

In [ ]:
import torch
import time
import gc

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False

try:
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
except ImportError:
    HAS_KEOPS = False

# --- CONFIG ---
N = 2**18        # 262.144
D = 3             
r0 = .1
c = 1.0
device = 'cuda'

print(f"--- FINAL BENCHMARK ---")
print(f"N={N}, Device={device}")

# ============================================================================
# TRITON FINAL (1D + PIPELINING)
# ============================================================================
if HAS_TRITON:
    @triton.jit
    def triton_kernel_final(
        ptr_q, ptr_forces,
        N: tl.int32, sigma_sq: tl.float32, V0: tl.float32,
        BLOCK_SIZE: tl.constexpr
    ):
        pid = tl.program_id(axis=0)
        
        # Konstanten
        prefactor = V0 / sigma_sq
        inv_width = -1.0 / (2.0 * sigma_sq)
        
        # Offsets
        off_x = pid
        off_y = N + pid
        off_z = 2 * N + pid
        
        # Eigene Position laden
        q_i_x = tl.load(ptr_q + off_x)
        q_i_y = tl.load(ptr_q + off_y)
        q_i_z = tl.load(ptr_q + off_z)
        
        acc_fx = 0.0; acc_fy = 0.0; acc_fz = 0.0
        
        # Loop über Nachbarn
        # Annahme: N % BLOCK_SIZE == 0
        for j_start in range(0, N, BLOCK_SIZE):
            offs = tl.arange(0, BLOCK_SIZE)
            j_idxs = j_start + offs
            
            # Vector Load
            q_j_x = tl.load(ptr_q + j_idxs)
            q_j_y = tl.load(ptr_q + N + j_idxs)
            q_j_z = tl.load(ptr_q + 2 * N + j_idxs)
            
            dx = q_i_x - q_j_x
            dy = q_i_y - q_j_y
            dz = q_i_z - q_j_z
            
            r_sq = dx*dx + dy*dy + dz*dz
            
            # Maskiere Selbstinteraktion mit Infinity
            # Das ist billiger als Masken und Branching
            r_sq = tl.where(pid == j_idxs, float('inf'), r_sq)
            
            # Fast Math Approximation für exp? 
            # Triton nutzt standardmäßig schon PTX exp2, das ist sehr schnell.
            val = tl.exp(inv_width * r_sq)
            force_mag = val * prefactor
            
            acc_fx += tl.sum(dx * force_mag)
            acc_fy += tl.sum(dy * force_mag)
            acc_fz += tl.sum(dz * force_mag)

        tl.store(ptr_forces + off_x, acc_fx)
        tl.store(ptr_forces + off_y, acc_fy)
        tl.store(ptr_forces + off_z, acc_fz)

    def run_triton_final(q, sigma, V0):
        # TUNING PARAMETER
        # BLOCK_SIZE=512 oder 1024 ist oft besser für Pipelining als 2048
        # num_warps=8: Mehr Threads um Latenz zu verstecken
        # num_stages=3: Lade die nächsten 3 Blöcke schon während des Rechnens
        BS = 1024
        WARPS = 8
        STAGES = 3
        
        if not q.is_contiguous(): q = q.contiguous()
        forces = torch.empty_like(q)
        grid = (N,)
        
        triton_kernel_final[grid](
            q, forces, 
            N, sigma**2, V0, 
            BLOCK_SIZE=BS, 
            num_warps=WARPS,
            num_stages=STAGES
        )
        return forces, 0

# ============================================================================
# KEOPS
# ============================================================================
def keops_NxD(positions, r0, c):
    N = positions.shape[0]; r0_2 = r0**2
    x_i = LazyTensor(positions[:, None, :])
    x_j = LazyTensor(positions[None, :, :])
    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term 
    forces = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces, 0

def keops_DxN(positions, r0, c):
    pos_NxD = positions.T.contiguous() 
    N = pos_NxD.shape[0]; r0_2 = r0**2
    x_i = LazyTensor(pos_NxD[:, None, :])
    x_j = LazyTensor(pos_NxD[None, :, :])
    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term 
    forces_NxD = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces_NxD.T, 0

# ============================================================================
# MAIN
# ============================================================================
q_NxD = torch.randn(N, D, device=device, dtype=torch.float32)
q_DxN = q_NxD.T.contiguous() 

algos = []
if HAS_TRITON:
    algos.append(("Triton Final (DxN)", run_triton_final, q_DxN, "DxN"))

if HAS_KEOPS:
    algos.append(("KeOps        (NxD)", keops_NxD,        q_NxD, "NxD"))
    algos.append(("KeOps        (DxN)", keops_DxN,        q_DxN, "DxN"))

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35}")
print("-" * 90)

# Warmup
print("Warmup...")
for name, func, data, layout in algos:
    try: func(data, r0, c)
    except: pass
print("Warmup fertig.\n")

# Run
for name, func, data, layout in algos:
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    
    start = time.perf_counter()
    f, p = func(data, r0, c)
    torch.cuda.synchronize()
    duration = time.perf_counter() - start

    f0 = f[:, 0] if layout == "DxN" else f[0, :]
    f0_str = f"[{f0[0]:.5f}, {f0[1]:.5f}, {f0[2]:.5f}]"
    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35}")

print("-" * 90)

--- FINAL BENCHMARK ---
N=262144, Device=cuda
ALGORITHMUS          | ZEIT (s) | KRAFT TEILCHEN 0                   
------------------------------------------------------------------------------------------
Warmup...
Warmup fertig.

Triton Final (DxN)   | 0.35871  | [-4.72458, 5.07956, 8.79864]       
KeOps        (NxD)   | 0.08429  | [-4.72458, 5.07956, 8.79865]       
KeOps        (DxN)   | 0.08498  | [-4.72458, 5.07956, 8.79865]       
------------------------------------------------------------------------------------------


In [ ]:
import torch
import time
import math
import gc

# Versuche Triton zu importieren
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False
    print("WARNUNG: Triton konnte nicht importiert werden.")

# Versuche KeOps zu importieren
try:
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
except ImportError:
    HAS_KEOPS = False

# --- KONFIGURATION ---
N = 2**17        # 16.384 Teilchen (Startwert)
D = 3             
r0 = .1
c = 1.0
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"--- BENCHMARK ---")
print(f"N={N}, Device={device}")

# ============================================================================
# TRITON KERNEL 1: Optimized (Unsafe - nur für N = Vielfaches von Block)
# ============================================================================
if HAS_TRITON:
    @triton.jit
    def triton_kernel_DxN_optimized(
        ptr_q, ptr_forces,
        N: tl.int32, sigma_sq: tl.float32, V0: tl.float32,
        BLOCK_SIZE: tl.constexpr
    ):
        # Program ID = Teilchen Index i
        pid = tl.program_id(axis=0)
        
        prefactor = V0 / sigma_sq
        inv_width = -1.0 / (2.0 * sigma_sq)
        
        off_x = pid
        off_y = N + pid
        off_z = 2 * N + pid
        
        q_i_x = tl.load(ptr_q + off_x)
        q_i_y = tl.load(ptr_q + off_y)
        q_i_z = tl.load(ptr_q + off_z)
        
        acc_fx = 0.0; acc_fy = 0.0; acc_fz = 0.0
        
        # Annahme: N % BLOCK_SIZE == 0 -> Keine Masken nötig
        for j_start in range(0, N, BLOCK_SIZE):
            offs = tl.arange(0, BLOCK_SIZE)
            j_idxs = j_start + offs
            
            q_j_x = tl.load(ptr_q + j_idxs)
            q_j_y = tl.load(ptr_q + N + j_idxs)
            q_j_z = tl.load(ptr_q + 2 * N + j_idxs)
            
            dx = q_i_x - q_j_x
            dy = q_i_y - q_j_y
            dz = q_i_z - q_j_z
            
            r_sq = dx*dx + dy*dy + dz*dz
            
            # Keine Selbstinteraktions-Maske (da Gauß Kraft = 0 bei r=0)
            
            val = tl.exp(inv_width * r_sq)
            force_mag = val * prefactor
            
            acc_fx += tl.sum(dx * force_mag)
            acc_fy += tl.sum(dy * force_mag)
            acc_fz += tl.sum(dz * force_mag)

        tl.store(ptr_forces + off_x, acc_fx)
        tl.store(ptr_forces + off_y, acc_fy)
        tl.store(ptr_forces + off_z, acc_fz)

    def run_triton_optimized(q, sigma, V0):
        if not q.is_contiguous(): q = q.contiguous()
        D, N = q.shape
        forces = torch.empty_like(q)
        grid = (N,)
        # Blocksize kann hier N sein, wenn N klein ist, sonst besser 1024/2048
        # Für den Benchmark mit N=2**14 nehmen wir 2048
        triton_kernel_DxN_optimized[grid](q, forces, N, sigma**2, V0, BLOCK_SIZE=2048, num_warps=4)
        return forces, 0

# ============================================================================
# TRITON KERNEL 2: Safe (Mit Masken - Funktioniert IMMER)
# ============================================================================
if HAS_TRITON:
    @triton.jit
    def triton_kernel_DxN_safe(
        ptr_q, ptr_forces,
        N: tl.int32, sigma_sq: tl.float32, V0: tl.float32,
        BLOCK_SIZE: tl.constexpr
    ):
        pid = tl.program_id(axis=0)
        
        # Sicherheitscheck: Falls Grid größer als N ist (passiert beim Aufrunden)
        if pid >= N:
            return

        prefactor = V0 / sigma_sq
        inv_width = -1.0 / (2.0 * sigma_sq)
        
        off_x = pid
        off_y = N + pid
        off_z = 2 * N + pid
        
        q_i_x = tl.load(ptr_q + off_x)
        q_i_y = tl.load(ptr_q + off_y)
        q_i_z = tl.load(ptr_q + off_z)
        
        acc_fx = 0.0; acc_fy = 0.0; acc_fz = 0.0
        
        # Loop: Wir iterieren durch ALLES, maskieren aber den Überhang am Ende
        for j_start in range(0, N, BLOCK_SIZE):
            offs = tl.arange(0, BLOCK_SIZE)
            j_idxs = j_start + offs
            
            # --- MASKE FÜR KRUMME N ---
            # True für gültige Teilchen, False für Überhang
            mask = j_idxs < N
            
            # Safe Load: Lädt 0.0 wo mask False ist, damit kein Speicherzugriffsfehler passiert
            q_j_x = tl.load(ptr_q + j_idxs, mask=mask, other=0.0)
            q_j_y = tl.load(ptr_q + N + j_idxs, mask=mask, other=0.0)
            q_j_z = tl.load(ptr_q + 2 * N + j_idxs, mask=mask, other=0.0)
            
            dx = q_i_x - q_j_x
            dy = q_i_y - q_j_y
            dz = q_i_z - q_j_z
            
            r_sq = dx*dx + dy*dy + dz*dz
            
            # Trick: Setze r_sq auf unendlich für ungültige Teilchen (mask=False).
            # exp(-inf) = 0. Damit tragen sie nichts zur Kraft bei.
            r_sq = tl.where(mask, r_sq, float('inf'))
            
            val = tl.exp(inv_width * r_sq)
            force_mag = val * prefactor
            
            acc_fx += tl.sum(dx * force_mag)
            acc_fy += tl.sum(dy * force_mag)
            acc_fz += tl.sum(dz * force_mag)

        tl.store(ptr_forces + off_x, acc_fx)
        tl.store(ptr_forces + off_y, acc_fy)
        tl.store(ptr_forces + off_z, acc_fz)

    def run_triton_safe(q, sigma, V0):
        if not q.is_contiguous(): q = q.contiguous()
        D, N = q.shape
        forces = torch.empty_like(q)
        
        # Grid: Wir müssen aufrunden, falls N kein Vielfaches von WARP ist (macht Triton meist selbst)
        grid = (N,)
        
        # BLOCK_SIZE=2048 ist ein guter Allrounder
        triton_kernel_DxN_safe[grid](q, forces, N, sigma**2, V0, BLOCK_SIZE=2048, num_warps=4)
        return forces, 0

# ============================================================================
# KEOPS
# ============================================================================
def keops_DxN(positions, r0, c):
    pos_NxD = positions.T.contiguous() 
    N = pos_NxD.shape[0]; r0_2 = r0**2

    x_i = LazyTensor(pos_NxD[:, None, :])
    x_j = LazyTensor(pos_NxD[None, :, :])

    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term 

    forces_NxD = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces_NxD.T, 0

# ============================================================================
# MAIN
# ============================================================================
q_NxD = torch.randn(N, D, device=device, dtype=dtype)
q_DxN = q_NxD.T.contiguous() # (3, N) layout

algos = []

if HAS_TRITON:
    algos.append(("Triton Opt (DxN)", run_triton_optimized, q_DxN, "DxN"))
    algos.append(("Triton Safe(DxN)", run_triton_safe,      q_DxN, "DxN")) # Hier eingefügt
    
if HAS_KEOPS:
    algos.append(("KeOps     (DxN)", keops_DxN,             q_DxN, "DxN"))

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35}")
print("-" * 90)

# 1. WARMUP
print("Warmup läuft...")
for name, func, data, layout in algos:
    try:
        func(data, r0, c)
    except Exception as e:
        print(f"Warmup Fehler bei {name}: {e}")
print("Warmup fertig.\n")

# 2. RUN
for name, func, data, layout in algos:
    # Cleanup
    gc.collect()
    torch.cuda.empty_cache()
    if device == 'cuda': torch.cuda.synchronize()

    start = time.perf_counter()
    f, p = func(data, r0, c)
    if device == 'cuda': torch.cuda.synchronize()
    duration = time.perf_counter() - start

    if layout == "DxN": f0 = f[:, 0]
    else: f0 = f[0, :]
    
    f0_str = f"[{f0[0]:.5f}, {f0[1]:.5f}, {f0[2]:.5f}]"
    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35}")

print("-" * 90)

--- BENCHMARK ---
N=131072, Device=cuda
ALGORITHMUS          | ZEIT (s) | KRAFT TEILCHEN 0                   
------------------------------------------------------------------------------------------
Warmup läuft...
Warmup fertig.

Triton Opt (DxN)     | 0.03278  | [57.86103, 22.60944, 2.13990]      
Triton Safe(DxN)     | 0.03940  | [57.86103, 22.60944, 2.13990]      
KeOps     (DxN)      | 0.02295  | [57.86104, 22.60945, 2.13991]      
------------------------------------------------------------------------------------------


In [ ]:
import torch
import time
torch.set_float32_matmul_precision('high')
# PyKeOps Check
try:
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
except ImportError:
    HAS_KEOPS = False
    print("KeOps nicht gefunden. Überspringe...")

# --- KONFIGURATION ---
N = 2**17         # 131.072 Teilchen
D = 3             
CHUNK = 2**11     # 2048 (Sicherer Wert für alle Methoden)
r0 = .1
c = 1
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"--- BENCHMARK ---")
print(f"N={N}, CHUNK={CHUNK}, Device={device}\n")

# ============================================================================
# 1. BROADCASTING (Speicherintensiv!)
# ============================================================================
@torch.compile()
def chunked_DxN(q, sigma, V0, chunk=CHUNK):
    # Benötigt (D, Chunk, N) Speicher -> OOM Gefahr bei großem Chunk!
    D, N = q.shape
    sigma_sq = sigma**2
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    
    forces = torch.empty_like(q)

    # Wir nutzen Static Shapes (kein min), N muss durch Chunk teilbar sein
    for i in range(0, N, chunk):
        q_chunk = q[:, i:i+chunk] 

        # 3D Tensor Explosion hier:
        diff = q_chunk[:, :, None] - q[:, None, :] 

        r_sq = (diff**2).sum(dim=0)
        exp_term = torch.exp(inv_width * r_sq)
        force_magnitude = exp_term * prefactor_force
        force_vecs = diff * force_magnitude[None, :, :]

        forces[:, i:i+chunk] = force_vecs.sum(dim=2)

    return forces, 0

@torch.compile()
def chunked_NxD(q, r0, c, chunk=CHUNK):
    N, D = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.empty_like(q)
    
    for i in range(0, N, chunk):
        q_chunk = q[i:i+chunk] 
        diff = q_chunk[:, None, :] - q[None, :, :] # OOM Gefahr
        r_sq = (diff**2).sum(dim=2)
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        force_vecs = diff * (exp_term * factor)[:, :, None]
        forces[i:i+chunk] = force_vecs.sum(dim=1)

    return forces, 0

# ============================================================================
# 2. FAST MATMUL (Speichereffizient & Schnell!)
# ============================================================================
@torch.compile()
def fast_chunked_DxN(q, sigma, V0, chunk=CHUNK):
    """
    Vermeidet den 3D-Tensor durch Matrix-Multiplikation.
    Kann auch mit größerem Chunk (z.B. 8192) laufen ohne OOM.
    """
    D, N = q.shape
    sigma_sq = sigma**2
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    
    q_sq = (q**2).sum(dim=0)
    forces = torch.empty_like(q)

    for i in range(0, N, chunk):
        # Slices
        q_chunk = q[:, i:i+chunk]
        q_sq_chunk = q_sq[i:i+chunk]
        
        # 1. ABSTÄNDE via MatMul (Nutzt Tensor Cores)
        term_dot = torch.matmul(q_chunk.T, q)
        
        r_sq = q_sq_chunk[:, None] + q_sq[None, :] - 2 * term_dot
        r_sq = torch.clamp(r_sq, min=0.0)
        
        # 2. KRAFT
        exp_term = torch.exp(inv_width * r_sq)
        force_magnitude = exp_term * prefactor_force
        
        sum_weights = force_magnitude.sum(dim=1)
        term_push = q_chunk * sum_weights[None, :]
        term_pull = torch.matmul(q, force_magnitude.T)
        
        forces[:, i:i+chunk] = term_push - term_pull

    return forces, 0

# ============================================================================
# 3. KEOPS
# ============================================================================
def keops_NxD(positions, r0, c):
    N = positions.shape[0]; r0_2 = r0**2
    x_i = LazyTensor(positions[:, None, :])
    x_j = LazyTensor(positions[None, :, :])
    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term 
    forces = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces, 0

# ============================================================================
# MAIN
# ============================================================================

# Daten generieren
q_NxD = torch.randn(N, D, device=device, dtype=dtype)
q_DxN = q_NxD.T.contiguous()

# Algorithmen Liste
algos = [
    # Wir testen die Broadcasting Varianten mit dem reduzierten Chunk (2048)
    ("Chunked (DxN)", chunked_DxN, q_DxN, "DxN"),
   # ("Chunked (NxD)", chunked_NxD, q_NxD, "NxD"),
    
    # Die optimierte Variante (Könnte auch Chunk=8192 vertragen)
    ("Fast+Comp (DxN)", fast_chunked_DxN, q_DxN, "DxN"),
]

if HAS_KEOPS:
    algos.append(("KeOps   (NxD)", keops_NxD, q_NxD, "NxD"))

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35}")
print("-" * 90)

# Warmup für torch.compile
print("Warmup läuft (für Compiler)...")
with torch.no_grad():
    fast_chunked_DxN(q_DxN, r0, c, chunk=CHUNK)
    chunked_DxN(q_DxN, r0, c, chunk=CHUNK)
print("Warmup fertig.\n")

for name, func, data, layout in algos:
    cleanup()
    if device == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()

    f, p = func(data, r0, c)

    if device == 'cuda': torch.cuda.synchronize()
    duration = time.perf_counter() - start

    if layout == "DxN":
        f0 = f[:, 0]
    else:
        f0 = f[0, :]

    f0_str = f"[{f0[0]:.5f}, {f0[1]:.5f}, {f0[2]:.5f}]"
    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35}")

print("-" * 90)

--- BENCHMARK ---
N=131072, CHUNK=2048, Device=cuda

ALGORITHMUS          | ZEIT (s) | KRAFT TEILCHEN 0                   
------------------------------------------------------------------------------------------
Warmup läuft (für Compiler)...
Warmup fertig.

Chunked (DxN)        | 3.35153  | [-27.55620, -2.52688, -37.94093]   
Fast+Comp (DxN)      | 4.29377  | [-27.52970, -2.47296, -37.87663]   
KeOps   (NxD)        | 0.02333  | [-27.55621, -2.52687, -37.94092]   
------------------------------------------------------------------------------------------


In [ ]:
D = 3
n_particles = 2**15

dtype = torch.float32
device_gpu = 'cuda'

N_REPEATS_GPU = 1

''' Initialization of position matrices  '''
q_DxN = torch.randn(D, n_particles, device=device_gpu, dtype=dtype)
q1_Dx1 = q_DxN[:,0]
delta = torch.empty_like(q_DxN)
abs_1xN = torch.empty(1, n_particles, device=device_gpu, dtype=dtype)

print(q1_Dx1)

delta = q_DxN - q1_Dx1[:, None]
sq_norm_1xN = ((q_DxN - q1_Dx1[:, None])**2).sum(dim=0)
#sq_norm_1xN_torch = (q_DxN - q1_Dx1[:, None]).sqnorm2()


print(delta)
print(sq_norm_1xN)

tensor([-0.0865,  0.4251, -1.2045], device='cuda:0')
tensor([[ 0.0000,  0.5730,  1.2609,  ..., -0.8362,  0.2374, -0.7449],
        [ 0.0000, -1.4491,  0.8686,  ..., -0.3887,  0.8051, -2.1121],
        [ 0.0000,  0.8591,  1.4099,  ...,  1.4162,  1.3358,  1.4360]],
       device='cuda:0')
tensor([0.0000, 3.1664, 4.3321,  ..., 2.8560, 2.4888, 7.0776], device='cuda:0')


In [ ]:
import torch
import time

# --- KONFIGURATION ---
N = 2**17       # Teilchenzahl
D = 3             # Dimensionen
CHUNK = 2**13     # Blockgröße
r0 = .1
c = 1
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'




print(f"--- BENCHMARK ---")
print(f"N={N}, Device={device}\n")

# ============================================================================
# LOOP IMPLEMENTIERUNGEN
# ============================================================================
def loop_DxN(q, r0, c):
    D, N = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.zeros_like(q)
    pot_acc = 0.0

    for i in range(N):
        # q: (D, N)

        # ÄNDERUNG HIER:
        # 1. q[:, i] holt den i-ten Vektor -> Shape (D,) (Dimension verloren)
        # 2. [:, None] fügt eine leere Dimension hinten an -> Shape (D, 1)
        q_i = q[:, i][:, None]

        # Broadcasting: (D, 1) - (D, N) -> (D, N)
        diff = q_i - q ## zu diff_i 

        r_sq = (diff**2).sum(dim=0) # Summe über Dimension 0 (D) -> (N,)
        exp_term = torch.exp(-r_sq / (2 * r0_2))

        # Broadcasting: (D, N) * (1, N) -> (D, N)
        force_vecs = diff * (exp_term * factor)[None, :]

        forces[:, i] = force_vecs.sum(dim=1)
        pot_acc += (c * exp_term).sum()

    return forces, 0.5 * (pot_acc - N * c)

def loop_NxD(q, r0, c):
    N, D = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.zeros_like(q)
    pot_acc = 0.0

    for i in range(N):
        # ÄNDERUNG HIER:
        # 1. q[i] holt den i-ten Vektor -> Shape (D,) (Dimension verloren)
        # 2. [None, :] fügt eine leere Dimension vorne an -> Shape (1, D)
        q_i = q[i][None, :]

        # Broadcasting: (1, D) - (N, D) -> (N, D)
        diff = q_i - q

        r_sq = (diff**2).sum(dim=1) # Summe über Dimension 1 (D) -> (N,)
        exp_term = torch.exp(-r_sq / (2 * r0_2))

        # Broadcasting: (N, D) * (N, 1) -> (N, D)
        force_vecs = diff * (exp_term * factor)[:, None]

        forces[i] = force_vecs.sum(dim=0)
        pot_acc += (c * exp_term).sum()

    return forces, 0.5 * (pot_acc - N * c)

# ============================================================================
# VOLLE MATRIX (NAIVE)
# ============================================================================
def naive_DxN(q, r0, c):
    D, N = q.shape
    r0_2 = r0**2; factor = c / r0_2

    # Broadcasting Matrix-Erstellung:
    # q[:, :, None] -> (D, N, 1) : Teilchen als Spaltenvektoren in der Tiefe
    # q[:, None, :] -> (D, 1, N) : Teilchen als Zeilenvektoren in der Tiefe
    # Differenz: (D, N, 1) - (D, 1, N) -> (D, N, N)
    # Ergebnis ist eine 3D-Matrix mit allen Paar-Differenzen.
    diff = q[:, :, None] - q[:, None, :]

    r_sq = (diff**2).sum(dim=0) # (D, N, N) -> (N, N) : Quadrierte Abstände
    exp_term = torch.exp(-r_sq / (2 * r0_2))

    # Broadcasting Kraft:
    # diff: (D, N, N)
    # exp_term: (N, N) -> wird via [None, :, :] zu (1, N, N)
    # Elementweise Multiplikation über die Dimension D.
    force_vecs = diff * (exp_term * factor)[None, :, :]

    forces = force_vecs.sum(dim=2) # Summiere Einflüsse aller j auf i -> (D, N)
    pot = 0.5 * ((c * exp_term).sum() - N * c)
    return forces, pot

def naive_NxD(q, r0, c):
    N, D = q.shape
    r0_2 = r0**2; factor = c / r0_2

    # Broadcasting Matrix-Erstellung:
    # q[:, None, :] -> (N, 1, D) : Jeder Vektor ist eine "Zeile" in der Tiefe
    # q[None, :, :] -> (1, N, D) : Jeder Vektor ist eine "Spalte" in der Tiefe
    # Differenz: (N, 1, D) - (1, N, D) -> (N, N, D)
    diff = q[:, None, :] - q[None, :, :]

    r_sq = (diff**2).sum(dim=2) # (N, N, D) -> (N, N)
    exp_term = torch.exp(-r_sq / (2 * r0_2))

    # Broadcasting Kraft:
    # diff: (N, N, D)
    # exp_term: (N, N) -> wird via [:, :, None] zu (N, N, 1)
    force_vecs = diff * (exp_term * factor)[:, :, None]

    forces = force_vecs.sum(dim=1) # (N, D)
    pot = 0.5 * ((c * exp_term).sum() - N * c)
    return forces, pot

# ============================================================================
# CHUNKED IMPLEMENTIERUNGEN
# ============================================================================
@torch.compile()
def chunked_DxN(q, sigma, V0, chunk=CHUNK):
    D, N = q.shape
    sigma_sq = sigma**2
    
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    
    forces = torch.empty_like(q)

    for i in range(0, N, chunk):
        
        q_chunk = q[:, i:i+chunk] # (D, ChunkSize)

        diff = q_chunk[:, :, None] - q[:, None, :]

        r_sq = (diff**2).sum(dim=0)
        
        exp_term = torch.exp(inv_width * r_sq)
        
        # Berechnung der skalaren Kraftstärke
        force_magnitude = exp_term * prefactor_force
        
        # Multiplikation mit Richtungsvektor
        force_vecs = diff * force_magnitude[None, :, :]

        forces[:, i:i+chunk] = force_vecs.sum(dim=2)

    return forces, 0

import torch

# Nutze "reduce-overhead" oder "max-autotune" für maximale Performance
@torch.compile()
def fast_chunked_DxN(q, sigma, V0, chunk=CHUNK):
    """
    Optimiert mit Matrix-Multiplikation & torch.compile.
    Vermeidet den Speicherfresser (D, Chunk, N).
    """
    D, N = q.shape
    sigma_sq = sigma**2
    
    # Vorberechnungen
    prefactor_force = V0 / sigma_sq
    inv_width = -1 / (2 * sigma_sq)
    
    # |q|^2 für alle Teilchen vorausberechnen (Shape: N)
    # Das spart Rechenzeit in der Schleife
    q_sq = (q**2).sum(dim=0)
    
    forces = torch.empty_like(q)

    for i in range(0, N, chunk):
        end = min(i+chunk, N)
        
        # Slices (Views)
        q_chunk = q[:, i:end]       # (D, Chunk)
        q_sq_chunk = q_sq[i:end]    # (Chunk,)
        
        # 1. ABSTÄNDE via MatMul (Der Turbo!)
        # Formel: r^2 = |a|^2 + |b|^2 - 2<a,b>
        # (Chunk, D) @ (D, N) -> (Chunk, N)
        term_dot = torch.matmul(q_chunk.T, q)
        
        # Broadcasting: (Chunk, 1) + (1, N) - (Chunk, N)
        r_sq = q_sq_chunk[:, None] + q_sq[None, :] - 2 * term_dot
        
        # Numerische Stabilität (negative Nullen verhindern)
        r_sq = torch.clamp(r_sq, min=0.0)
        
        # 2. EXPONENT
        exp_term = torch.exp(inv_width * r_sq) # (Chunk, N)
        
        # Skalieren für die Kraft
        # (Chunk, N)
        force_magnitude = exp_term * prefactor_force
        
        # 3. KRAFT (Zerlegung in Push & Pull)
        # Statt diff * magnitude nutzen wir Matrix-Operationen:
        # F_i = q_i * sum(W) - sum(q_j * W)
        
        # Term 1: Abstoßung vom eigenen Ort
        sum_weights = force_magnitude.sum(dim=1)     # (Chunk,)
        term_push = q_chunk * sum_weights[None, :]   # (D, Chunk)
        
        # Term 2: Einfluss der Nachbarn
        # (D, N) @ (N, Chunk) -> (D, Chunk)
        term_pull = torch.matmul(q, force_magnitude.T)
        
        forces[:, i:end] = term_push - term_pull

    return forces, 0

@torch.compile()
def chunked_NxD(q, r0, c, chunk=CHUNK):
    N, D = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.empty_like(q)
    pot_acc = 0.0

    for i in range(0, N, chunk):
        end = min(i+chunk, N)
        q_chunk = q[i:end] # (ChunkSize, D)

        # Broadcasting Chunk:
        # q_chunk[:, None, :] -> (ChunkSize, 1, D)
        # q[None, :, :]       -> (1, N, D)
        # Ergebnis: (ChunkSize, N, D)
        diff = q_chunk[:, None, :] - q[None, :, :]

        r_sq = (diff**2).sum(dim=2)
        exp_term = torch.exp(-r_sq / (2 * r0_2))

        # Broadcasting: (ChunkSize, N, D) * (ChunkSize, N, 1) -> (ChunkSize, N, D)
        force_vecs = diff * (exp_term * factor)[:, :, None]

        forces[i:end] = force_vecs.sum(dim=1)
        pot_acc += (c * exp_term).sum()

    return forces, 0.5 * (pot_acc - N * c)

# ============================================================================
# PYKEOPS
# ============================================================================
def keops_NxD(positions, r0, c):
    N = positions.shape[0]; r0_2 = r0**2

    # LazyTensors erstellen symbolische Matrizen, keine echten Daten im RAM
    x_i = LazyTensor(positions[:, None, :])
    x_j = LazyTensor(positions[None, :, :])

    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term # Symbolische Matrix (N, N)

    # sum_reduction triggert die echte Berechnung
    #pot = 0.5 * (potential_ij.sum_reduction(axis=1).sum() - N*c)
    forces = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces, 0#pot

# ============================================================================
# MAIN RUNNER
# ============================================================================

# Daten
q_NxD = torch.randn(N, D, device=device, dtype=dtype)
q_DxN = q_NxD.T.contiguous()

# Liste der zu testenden Funktionen
algos = [
   # ("Naive   (DxN)", naive_DxN,   q_DxN, "DxN"),
   # ("Naive   (NxD)", naive_NxD,   q_NxD, "NxD"),
   # ("Loop    (DxN)", loop_DxN,    q_DxN, "DxN"),
    #("Loop    (NxD)", loop_NxD,    q_NxD, "NxD"),
    ("Chunked (DxN)", chunked_DxN, q_DxN, "DxN"),
    ("Chunked (NxD)", chunked_NxD, q_NxD, "NxD"),
   # ("Fast+Comp (DxN)", fast_chunked_DxN, q_DxN, "DxN"),
    ("KeOps   (NxD)", keops_NxD, q_NxD, "NxD")
]

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35} | {'POTENZIAL'}")
print("-" * 90)

for name, func, data, layout in algos:
    cleanup()

    # Synchronize vor Start
    if device == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()

    f, p = func(data, r0, c)

    # Synchronize nach Ende
    if device == 'cuda': torch.cuda.synchronize()
    duration = time.perf_counter() - start

    # Kraft von Teilchen 0 extrahieren
    if layout == "DxN":
        # Form ist (3, N) -> wir nehmen [:, 0] -> (3,)
        f0 = f[:, 0]
    else:
        # Form ist (N, 3) -> wir nehmen [0, :] -> (3,)
        f0 = f[0, :]

    # Formatierung für Print (nur 3 Nachkommastellen für Lesbarkeit)
    f0_str = f"[{f0[0]:.5f}, {f0[1]:.5f}, {f0[2]:.5f}]"

    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35} | {p:.4e}")

print("-" * 90)

--- BENCHMARK ---
N=131072, Device=cuda

ALGORITHMUS          | ZEIT (s) | KRAFT TEILCHEN 0                    | POTENZIAL
------------------------------------------------------------------------------------------


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 47.40 GiB of which 3.12 GiB is free. Process 5199 has 264.69 MiB memory in use. Process 31934 has 4.31 GiB memory in use. Including non-PyTorch memory, this process has 37.17 GiB memory in use. Of the allocated memory 36.02 GiB is allocated by PyTorch, and 26.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
asd# ============================================================================
# 4. SCALING BENCHMARK & PLOTTING (NEU)
# ============================================================================
import matplotlib.pyplot as plt

print("\n" + "="*50)
print("STARTING SCALING BENCHMARK (N vs Time)")
print("="*50)

# 1. Definition der zu testenden N-Werte
# Tipp: Bei "Naive" Implementierungen vorsichtig sein (Max N ca. 10.000-20.000 bei viel VRAM)
# Für Chunked/KeOps kann man höher gehen.
N_values = [2**i for i in range(10, 19)]

# Dictionary zum Speichern der Ergebnisse: {'AlgoName': [Zeit_N1, Zeit_N2, ...]}
results = {name: [] for name, _, _, _ in algos}

# 2. Benchmark Loop
for n_curr in N_values:
    print(f"Benchmarking N = {n_curr}")

    # Neue Daten für dieses N generieren
    q_NxD_curr = torch.randn(n_curr, D, device=device, dtype=dtype)
    q_DxN_curr = q_NxD_curr.T.contiguous()

    for name, func, _, layout in algos:
        # WICHTIG: Wir ignorieren die Daten aus der 'algos'-Liste (das war das alte N)
        # und wählen die neuen Daten passend zum Layout:
        data_curr = q_DxN_curr if layout == "DxN" else q_NxD_curr

        # Synchronize & Timing
        if device == 'cuda': torch.cuda.synchronize()
        start = time.perf_counter()

        try:
            func(data_curr, r0, c)

            if device == 'cuda': torch.cuda.synchronize()
            duration = time.perf_counter() - start
            results[name].append(duration)

        except Exception as e:
            print(f"  !! Fehler bei {name} mit N={n_curr}: {e}")
            results[name].append(None) # None markiert einen Fehlschlag (z.B. OutOfMemory)

# 3. Plotting
plt.figure(figsize=(10, 6))

for name, times in results.items():
    # Wir filtern fehlgeschlagene Läufe (None) heraus, damit der Plot nicht bricht
    valid_n = [n for n, t in zip(N_values, times) if t is not None]
    valid_t = [t for t in times if t is not None]

    if valid_t:
        plt.plot(valid_n, valid_t, marker='o', label=name)

plt.title(f'N-Body Benchmark Scaling (Device: {device})')
plt.xlabel('Anzahl der Teilchen (N)')
plt.ylabel('Zeit (s)')
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.legend()
plt.xscale('log') # Optional: 'log' für sehr große Bereiche
plt.yscale('log') # Optional: 'log' um Unterschiede besser zu sehen
plt.tight_layout()

# Plot anzeigen
plt.show()


STARTING SCALING BENCHMARK (N vs Time)
Benchmarking N = 1024
Benchmarking N = 2048
Benchmarking N = 4096


KeyboardInterrupt: 

In [ ]:
import torch

def pair_force_one_to_all_DxN(
    q_DxN: torch.Tensor,
    r0: float,
    c: float
) -> tuple[torch.Tensor, float]:
    """
    Berechnet paarweise Kräfte und Potenziale mit einem Gauß-Kern.
    Verwendet eine 'One-to-All' Iteration (ein Teilchen gegen den Rest)
    unter Beibehaltung des DxN (3xN) Speicher-Layouts.

    Args:
        q_DxN: Tensor der Form (D, N) -> Positionen
        r0: Effektive Reichweite des Potenzials
        c: Interaktionsstärke

    Returns:
        forces_DxN: Tensor der Form (D, N) -> Gesamtkraft auf jedes Teilchen
        total_potential: Skalar (float) -> Gesamtpotenzialenergie des Systems
    """

    D, n_particles = q_DxN.shape
    device = q_DxN.device
    dtype = q_DxN.dtype

    # Vorberechnungen
    r0_2 = r0**2
    factor_force = c / r0_2

    # Speicher für Ergebnisse
    forces_DxN = torch.zeros((D, n_particles), device=device, dtype=dtype)
    total_potential_acc = 0.0

    # --- Schleife über jedes Teilchen i ---
    for i in range(n_particles):
        # 1. Position von Teilchen i (Form: D, 1)
        q_i = q_DxN[:, i].unsqueeze(1)

        # 2. Differenzvektor zu ALLEN anderen Teilchen j (Form: D, N)
        # diff = r_j - r_i
        # Vektor zeigt von i nach j.
        diff = q_DxN - q_i

        # 3. Quadratischer Abstand (Form: N,)
        # Summe über Dimension 0 (x,y,z)
        r_sq_N = (diff**2).sum(dim=0)

        # 4. Gauß-Term berechnen
        exp_term = torch.exp(-r_sq_N / (2 * r0_2))

        # 5. Potenzial akkumulieren (inklusive Selbstinteraktion, wird später korrigiert)
        total_potential_acc += (c * exp_term).sum()

        # 6. Kraft berechnen
        # Kraft auf i durch j (abstoßend) zeigt in Richtung -(r_j - r_i) = r_i - r_j.
        # Unser diff ist (r_j - r_i).
        # Also Kraft ~ -diff * exp_term.
        # Da wir diff * exp_term rechnen, zeigt der Vektor von i nach j.
        # Wenn c > 0 (Abstoßung), muss Teilchen i weg von j gedrückt werden.
        # Das ist die Richtung -(r_j - r_i).
        # Im Code rechnen wir: force_vecs = diff * scalar.
        # Das zeigt von i nach j.
        # Um die Kraft AUF i zu haben, die von j WEG zeigt (nach "links"),
        # brauchen wir eigentlich das negative Vorzeichen, wenn diff = r_j - r_i ist.
        # Aber warte: Im Original-Snippet stand `force_vecs = diff * ...`
        # Wenn diff = r_j - r_i ist (Vektor nach rechts), und wir stoßen ab,
        # dann wird Teilchen i nach links gedrückt.
        # Die Kraft auf i ist F_i = sum( F_ij ).
        # F_ij (auf i durch j) ~ (r_i - r_j).
        # diff ist (r_j - r_i). Also ist (r_i - r_j) = -diff.
        # Das bedeutet, hier fehlt mathematisch ein Minuszeichen oder die Definition von diff
        # müsste (q_i - q_DxN) sein.
        # Da dein Original-Snippet `diff = q_DxN - q_i` (r_j - r_i) nutzt,
        # und dann `forces = diff * ...` addiert, würde die Kraft in Richtung j zeigen (Anziehung?).
        # Für Abstoßung bei c > 0 sollte es `q_i - q_DxN` sein oder `forces -= ...`.

        # KORREKTUR für physikalische Konsistenz (Abstoßung):
        # Wir nutzen hier diff = q_i - q_DxN (r_i - r_j), damit der Vektor von j weg zeigt.
        # Das ist effizienter als am Ende mal -1 zu rechnen.
        diff_for_force = -diff  # oder oben direkt q_i - q_DxN rechnen

        force_magnitude = exp_term * factor_force
        force_vecs = diff_for_force * force_magnitude.unsqueeze(0)

        # Gesamtkraft auf i speichern
        forces_DxN[:, i] = force_vecs.sum(dim=1)

    # --- Nachbearbeitung ---

    # 1. Selbstinteraktion abziehen (N * c)
    # 2. Faktor 0.5 wegen Doppelzählung (i-j und j-i)
    total_potential = 0.5 * (total_potential_acc - n_particles * c)

    return forces_DxN, total_potential

In [ ]:
import torch
import time

# --- Setup ---
D = 3
n_particles = 2**15  # 32768 Teilchen
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

r0 = 1.0
c = 10.0

print(f"Berechnung auf: {device}")
print(f"Anzahl Teilchen: {n_particles}")

# Daten initialisieren (DxN Layout)
q_DxN = torch.randn(D, n_particles, device=device, dtype=dtype)

# --- Berechnung Starten ---
# Synchronisieren, damit die Zeitmessung korrekt startet
if device == 'cuda':
    torch.cuda.synchronize()

start_time = time.perf_counter()

# Der Aufruf der Funktion für ALLE Teilchen
forces, potential = pair_force_one_to_all_DxN(q_DxN, r0, c)

# Warten bis GPU fertig ist
if device == 'cuda':
    torch.cuda.synchronize()

end_time = time.perf_counter()
duration = end_time - start_time

# --- Ergebnisse ---
print("-" * 30)
print(f"Dauer: {duration:.4f} Sekunden")
print(f"Gesamtpotenzial: {potential:.4e}")
print(f"Kräfte Tensor Form: {forces.shape}") # Sollte torch.Size([3, 32768]) sein
print("-" * 30)

Berechnung auf: cuda
Anzahl Teilchen: 32768


KeyboardInterrupt: 

In [ ]:
import torch
import time

# ============================================================================
# 1. Funktion für DxN Layout (3 x N) - Structure of Arrays
# ============================================================================
def pair_force_one_to_all_DxN(q_DxN, r0, c):
    D, n_particles = q_DxN.shape
    device = q_DxN.device
    dtype = q_DxN.dtype
    r0_2 = r0**2
    factor = c / r0_2

    forces = torch.zeros((D, n_particles), device=device, dtype=dtype)
    potential_acc = 0.0

    for i in range(n_particles):
        # q_i shape: (3, 1)
        q_i = q_DxN[:, i].unsqueeze(1)

        # diff = r_i - r_j (Vektor zeigt von j nach i)
        # Shape: (3, N)
        diff = q_i - q_DxN

        # r^2 berechnen (Summe über Dimension 0: x+y+z)
        # Hier muss die GPU im Speicher weit springen (stride)
        r_sq = (diff**2).sum(dim=0)

        exp_term = torch.exp(-r_sq / (2 * r0_2))
        potential_acc += (c * exp_term).sum()

        # Kraft auf i = Summe ( faktor * exp * (r_i - r_j) )
        # diff ist hier (r_i - r_j), also passt das Vorzeichen für Abstoßung
        force_vecs = diff * (exp_term * factor).unsqueeze(0)

        forces[:, i] = force_vecs.sum(dim=1)

    # Korrekturen
    total_potential = 0.5 * (potential_acc - n_particles * c)
    return forces, total_potential


# ============================================================================
# 2. Funktion für NxD Layout (N x 3) - Array of Structures
# ============================================================================
def pair_force_one_to_all_NxD(q_NxD, r0, c):
    n_particles, D = q_NxD.shape
    device = q_NxD.device
    dtype = q_NxD.dtype
    r0_2 = r0**2
    factor = c / r0_2

    forces = torch.zeros((n_particles, D), device=device, dtype=dtype)
    potential_acc = 0.0

    for i in range(n_particles):
        # q_i shape: (1, 3)
        q_i = q_NxD[i:i+1, :]

        # diff = r_i - r_j
        # Shape: (N, 3)
        diff = q_i - q_NxD

        # r^2 berechnen (Summe über Dimension 1: x+y+z)
        # HIER GEWINNT NxD: x,y,z liegen im Speicher nebeneinander!
        r_sq = (diff**2).sum(dim=1)

        exp_term = torch.exp(-r_sq / (2 * r0_2))
        potential_acc += (c * exp_term).sum()

        # Kraftberechnung
        # Broadcasting: (N, 3) * (N, 1)
        force_vecs = diff * (exp_term * factor).unsqueeze(1)

        # Summe über alle j (Dimension 0)
        forces[i] = force_vecs.sum(dim=0)

    # Korrekturen
    total_potential = 0.5 * (potential_acc - n_particles * c)
    return forces, total_potential


# ============================================================================
# 3. Test & Vergleich
# ============================================================================

# Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float32
r0 = 1.0
c = 1
# Reduzierte Teilchenzahl für den Loop-Test (damit es schnell läuft)
# N = 2**15 # (32768) -> Dauert in Python Loop ca 30-60 sek
N = 2**14    # Kleiner Test für sofortiges Ergebnis

print(f"--- Starte Vergleichstest auf {device} mit N={N} Teilchen ---")

# Zufallsdaten erzeugen (NxD Format)
q_NxD_ref = torch.randn(N, 3, device=device, dtype=dtype)
# Transponieren für DxN Format (wichtig: .contiguous() für fairen Speichervergleich)
q_DxN_ref = q_NxD_ref.T.contiguous()

# --- TEST 1: DxN (3xN) ---
if device == 'cuda': torch.cuda.synchronize()
start = time.perf_counter()
f_DxN, p_DxN = pair_force_one_to_all_DxN(q_DxN_ref, r0, c)
if device == 'cuda': torch.cuda.synchronize()
time_DxN = time.perf_counter() - start

# --- TEST 2: NxD (Nx3) ---
if device == 'cuda': torch.cuda.synchronize()
start = time.perf_counter()
f_NxD, p_NxD = pair_force_one_to_all_NxD(q_NxD_ref, r0, c)
if device == 'cuda': torch.cuda.synchronize()
time_NxD = time.perf_counter() - start

# ============================================================================
# 4. Ergebnisse
# ============================================================================

print(f"\nZeiten:")
print(f"DxN (3xN) Layout: {time_DxN:.5f} s")
print(f"NxD (Nx3) Layout: {time_NxD:.5f} s")

factor = time_DxN / time_NxD
print(f"-> NxD war {factor:.2f}x schneller/langsamer als DxN.")

print(f"\nKorrektheits-Check:")
# Wir müssen DxN Ergebnis transponieren, um es mit NxD zu vergleichen
diff_forces = (f_DxN.T - f_NxD).abs().max().item()
diff_pot = abs(p_DxN - p_NxD)

print(f"Maximaler Unterschied Kräfte:    {diff_forces:.2e}")
print(f"Unterschied Potenzial:           {diff_pot:.2e}")



In [ ]:
import torch
import time
import math
from numba import cuda, float32

# --- KONFIGURATION ---
N_GLOBAL = 155000  # Umbenannt, damit wir es nicht aus Versehen nutzen
D = 3
CHUNK = 2048
TPB = 256

r0 = .1
c = 1
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============================================================================
# 1. Chunked (Korrigiert: N dynamisch)
# ============================================================================
def chunked_NxD(q, r0, c, chunk=CHUNK):
    # FIX: N aus der aktuellen Eingabe lesen!
    current_N = q.shape[0]

    forces = torch.empty_like(q)
    pot_acc = 0.0
    r0_2 = r0**2; factor = c / r0_2

    for i in range(0, current_N, chunk):
        end = min(i+chunk, current_N)
        q_chunk = q[i:end]
        diff = q_chunk.unsqueeze(1) - q.unsqueeze(0)
        r_sq = (diff**2).sum(dim=2)
        exp_term = torch.exp(-r_sq / (2 * r0_2))

        force_vecs = diff * (exp_term * factor).unsqueeze(2)
        forces[i:end] = force_vecs.sum(dim=1)
        pot_acc += (c * exp_term).sum()

    return forces, 0.5 * (pot_acc - current_N * c)

# ============================================================================
# 2. Numba Global (Korrigiert)
# ============================================================================
@cuda.jit
def numba_global_kernel(q, forces, pot_arr, r0, c, N):
    i = cuda.grid(1)
    if i < N:
        xi = q[i, 0]; yi = q[i, 1]; zi = q[i, 2]
        fx = 0.0; fy = 0.0; fz = 0.0
        p = 0.0
        r0_2 = r0**2; factor = c / r0_2

        for j in range(N):
            dx = xi - q[j, 0]
            dy = yi - q[j, 1]
            dz = zi - q[j, 2]
            r2 = dx*dx + dy*dy + dz*dz
            val = math.exp(-r2 / (2*r0_2))

            f_val = val * factor
            fx += dx * f_val
            fy += dy * f_val
            fz += dz * f_val
            p += c * val

        forces[i, 0] = fx
        forces[i, 1] = fy
        forces[i, 2] = fz
        pot_arr[i] = p

def run_numba_global(q, r0, c):
    # FIX: N dynamisch bestimmen
    current_N = q.shape[0]
    forces = torch.zeros_like(q)
    pot_arr = torch.zeros(current_N, device=q.device, dtype=q.dtype)
    blocks = (current_N + TPB - 1) // TPB

    numba_global_kernel[blocks, TPB](q, forces, pot_arr, r0, c, current_N)
    return forces, 0.5 * (pot_arr.sum() - current_N * c)

# ============================================================================
# 3. Numba Shared (Korrigiert)
# ============================================================================
@cuda.jit
def numba_shared_kernel(q, forces, pot_arr, r0, c, N):
    tid = cuda.threadIdx.x
    i = cuda.grid(1)
    s_pos = cuda.shared.array((TPB, 3), float32)

    xi, yi, zi = 0.0, 0.0, 0.0
    if i < N:
        xi = q[i, 0]; yi = q[i, 1]; zi = q[i, 2]

    fx, fy, fz = 0.0, 0.0, 0.0
    p = 0.0
    r0_2 = r0**2; factor = c / r0_2

    for tile_start in range(0, N, TPB):
        load_idx = tile_start + tid
        if load_idx < N:
            s_pos[tid, 0] = q[load_idx, 0]
            s_pos[tid, 1] = q[load_idx, 1]
            s_pos[tid, 2] = q[load_idx, 2]
        else:
            s_pos[tid, 0] = 0.0; s_pos[tid, 1] = 0.0; s_pos[tid, 2] = 0.0

        cuda.syncthreads()

        valid_in_tile = TPB
        if tile_start + TPB > N:
            valid_in_tile = N - tile_start

        if i < N:
            for j in range(valid_in_tile):
                dx = xi - s_pos[j, 0]
                dy = yi - s_pos[j, 1]
                dz = zi - s_pos[j, 2]
                r2 = dx*dx + dy*dy + dz*dz
                val = math.exp(-r2 / (2*r0_2))
                f_val = val * factor
                fx += dx * f_val
                fy += dy * f_val
                fz += dz * f_val
                p += c * val

        cuda.syncthreads()

    if i < N:
        forces[i, 0] = fx; forces[i, 1] = fy; forces[i, 2] = fz
        pot_arr[i] = p

def run_numba_shared(q, r0, c):
    # FIX: N dynamisch bestimmen
    current_N = q.shape[0]
    forces = torch.zeros_like(q)
    pot_arr = torch.zeros(current_N, device=q.device, dtype=q.dtype)
    blocks = (current_N + TPB - 1) // TPB

    numba_shared_kernel[blocks, TPB](q, forces, pot_arr, r0, c, current_N)
    return forces, 0.5 * (pot_arr.sum() - current_N * c)

# ============================================================================
# MAIN RUNNER
# ============================================================================

# Restart Check: Wenn CUDA kaputt ist, müssen wir neu initialisieren
try:
    q = torch.randn(N_GLOBAL, D, device=device, dtype=dtype)
except Exception:
    print("!!! BITTE KERNEL NEU STARTEN (Runtime > Restart Session) !!!")
    raise

algos = [
    ("PyTorch Chunked", chunked_NxD),
    ("Numba (Global)",  run_numba_global),
    ("Numba (Shared)",  run_numba_shared)
]

print(f"{'ALGORITHMUS':<20} | {'ZEIT (s)':<8} | {'KRAFT TEILCHEN 0':<35} | {'POTENZIAL'}")
print("-" * 90)

print("Warming up JIT kernels...", end="")
for _, func in algos:
    # Jetzt sicher: Funktion liest N=100 aus dem Input-Slice
    func(q[:100], r0, c)
print(" Done.\n")

for name, func in algos:
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    start = time.perf_counter()
    f, p = func(q, r0, c)
    torch.cuda.synchronize()
    duration = time.perf_counter() - start

    f0 = f[0, :].cpu()
    f0_str = f"[{f0[0]:.4f}, {f0[1]:.4f}, {f0[2]:.4f}]"
    print(f"{name:<20} | {duration:.5f}  | {f0_str:<35} | {p:.4e}")

print("-" * 90)

ALGORITHMUS          | ZEIT (s) | KRAFT TEILCHEN 0                    | POTENZIAL
------------------------------------------------------------------------------------------
Warming up JIT kernels...

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


 Done.

PyTorch Chunked      | 15.24869  | [44.4616, 14.5328, -126.7986]       | 4.2119e+06
Numba (Global)       | 4.22097  | [44.4616, 14.5328, -126.7986]       | 4.2119e+06
Numba (Shared)       | 4.93869  | [44.4616, 14.5328, -126.7986]       | 4.2119e+06
------------------------------------------------------------------------------------------


In [ ]:
import torch
import time
import math
import sys
!pip install pykeops
# Versuche KeOps zu importieren (optional)
try:
    from pykeops.torch import LazyTensor
    has_keops = True
except ImportError:
    has_keops = False
    print("INFO: PyKeOps nicht installiert. Dieser Test wird übersprungen.\n")

# --- KONFIGURATION ---
N = 10        # Teilchenzahl (N=4096 ist sicher für Loop/Naive. Bei N>10k Loop deaktivieren!)
D = 3           # Dimensionen
CHUNK = 1024    # Blockgröße für Chunking
r0 = 1.0
c = 10.0
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"===========================================================")
print(f"   ULTIMATIVER N-BODY VERGLEICH (9 VARIANTEN)")
print(f"   N={N}, Device={device}, Chunksize={CHUNK}")
print(f"===========================================================\n")

# ============================================================================
# GRUPPE 1: MANUELLE IMPLEMENTIERUNGEN (Loop, Naive, Chunked)
# ============================================================================

# --- 1.1 Naive (O(N^2) Speicher) ---
def naive_DxN(q, r0, c):
    r0_2 = r0**2; factor = c / r0_2
    diff = q.unsqueeze(2) - q.unsqueeze(1)
    r_sq = (diff**2).sum(dim=0) # Bad Stride
    exp_term = torch.exp(-r_sq / (2 * r0_2))
    force_vecs = diff * (exp_term * factor).unsqueeze(0)
    forces = force_vecs.sum(dim=2)
    pot = 0.5 * ((c * exp_term).sum() - q.shape[1] * c)
    return forces, pot

def naive_NxD(q, r0, c):
    r0_2 = r0**2; factor = c / r0_2
    diff = q.unsqueeze(1) - q.unsqueeze(0)
    r_sq = (diff**2).sum(dim=2) # Good Stride
    exp_term = torch.exp(-r_sq / (2 * r0_2))
    force_vecs = diff * (exp_term * factor).unsqueeze(2)
    forces = force_vecs.sum(dim=1)
    pot = 0.5 * ((c * exp_term).sum() - q.shape[0] * c)
    return forces, pot

# --- 1.2 Loop (Python Overhead Hölle) ---
def loop_DxN(q, r0, c):
    D, N = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.zeros_like(q)
    pot_acc = 0.0
    for i in range(N):
        q_i = q[:, i:i+1]
        diff = q_i - q
        r_sq = (diff**2).sum(dim=0)
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[:, i] = (diff * (exp_term * factor).unsqueeze(0)).sum(dim=1)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)

def loop_NxD(q, r0, c):
    N, D = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.zeros_like(q)
    pot_acc = 0.0
    for i in range(N):
        q_i = q[i:i+1]
        diff = q_i - q
        r_sq = (diff**2).sum(dim=1)
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[i] = (diff * (exp_term * factor).unsqueeze(1)).sum(dim=0)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)

# --- 1.3 Manual Chunked (Der Goldstandard für Manuell) ---
def manual_chunked_DxN(q, r0, c, chunk):
    D, N = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.empty_like(q); pot_acc = 0.0
    for i in range(0, N, chunk):
        end = min(i+chunk, N)
        q_chunk = q[:, i:end]
        diff = q_chunk.unsqueeze(2) - q.unsqueeze(1)
        r_sq = (diff**2).sum(dim=0) # Bad Stride
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[:, i:end] = (diff * (exp_term * factor).unsqueeze(0)).sum(dim=2)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)

def manual_chunked_NxD(q, r0, c, chunk):
    N, D = q.shape
    r0_2 = r0**2; factor = c / r0_2
    forces = torch.empty_like(q); pot_acc = 0.0
    for i in range(0, N, chunk):
        end = min(i+chunk, N)
        q_chunk = q[i:end]
        diff = q_chunk.unsqueeze(1) - q.unsqueeze(0)
        r_sq = (diff**2).sum(dim=2) # Good Stride
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[i:end] = (diff * (exp_term * factor).unsqueeze(2)).sum(dim=1)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)

# ============================================================================
# GRUPPE 2: ADVANCED (Compiled & KeOps) - Alle NxD Layout
# ============================================================================

# --- 2.1 Torch Compile Simple ---
@torch.compile(mode="max-autotune")
def compiled_simple_NxD(positions, r0, c):
    N = positions.shape[0]; r0_2 = r0**2
    R_vec = positions[:, None, :] - positions[None, :, :]
    r_sq = torch.sum(R_vec**2, dim=2)
    exp_term = torch.exp(-r_sq / (2.0 * r0_2))
    potential_ij = c * exp_term
    forces = torch.sum(potential_ij.unsqueeze(2) * R_vec / r0_2, dim=1)
    total_potential = 0.5 * (potential_ij.sum() - N * c)
    return forces, total_potential

# --- 2.2 Torch Compile Chunked ---
@torch.compile(mode="max-autotune")
def _step_compiled(x_i_chunk, x_j_all, r0_2, c):
    R_vec = x_i_chunk[:, None, :] - x_j_all[None, :, :]
    r_sq = torch.sum(R_vec**2, dim=2)
    exp_term = (-r_sq / (2 * r0_2)).exp()
    pot_sum = torch.sum(c * exp_term, dim=1)
    forces = torch.sum((c * exp_term).unsqueeze(2) * R_vec / r0_2, dim=1)
    return forces, pot_sum
@torch.compile(mode="max-autotune")
def compiled_chunked_NxD(positions, r0, c, chunk_size):
    N = positions.shape[0]; r0_2 = r0**2
    forces_list = []; pot_list = []
    for x_i in positions.split(chunk_size):
        f, p = _step_compiled(x_i, positions, r0_2, c)
        forces_list.append(f); pot_list.append(p)
    forces = torch.cat(forces_list, dim=0)
    pot = 0.5 * (torch.cat(pot_list, dim=0).sum() - N * c)
    return forces, pot

# --- 2.3 KeOps ---
def keops_NxD(positions, r0, c):
    if not has_keops: return None, 0.0
    N = positions.shape[0]; r0_2 = r0**2
    x_i = LazyTensor(positions[:, None, :])
    x_j = LazyTensor(positions[None, :, :])
    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term
    pot = 0.5 * (potential_ij.sum_reduction(axis=1).sum() - N*c)
    forces = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces, pot

# ============================================================================
# MAIN BENCHMARK RUNNER
# ============================================================================

# Daten initialisieren (NxD ist Basis)
q_NxD = torch.randn(N, D, device=device, dtype=dtype)
q_DxN = q_NxD.T.contiguous()

# Liste definieren: (Name, Funktion, Input-Daten, Layout, Braucht Chunk Argument?)
algos = [
   # ("Naive           (DxN)", naive_DxN,            q_DxN, "DxN", False),
   # ("Naive           (NxD)", naive_NxD,            q_NxD, "NxD", False),
  #  ("Loop            (DxN)", loop_DxN,             q_DxN, "DxN", False),
   # ("Loop            (NxD)", loop_NxD,             q_NxD, "NxD", False),
    ("Manual Chunked  (DxN)", manual_chunked_DxN,   q_DxN, "DxN", True),
    ("Manual Chunked  (NxD)", manual_chunked_NxD,   q_NxD, "NxD", True),
   # ("Torch Simple    (@JIT)", compiled_simple_NxD, q_NxD, "NxD", False),
    ("Torch Chunked   (@JIT)", compiled_chunked_NxD, q_NxD, "NxD", True),
]

if has_keops:
    algos.append(("PyKeOps         (Sym)", keops_NxD, q_NxD, "NxD", False))

print(f"{'ALGORITHMUS':<25} | {'ZEIT (s)':<10} | {'KRAFT TEILCHEN 0 (X, Y, Z)':<40} | {'POTENTIAL'}")
print("-" * 100)

for name, func, data, layout, needs_chunk in algos:
    try:
        # --- 1. Warm-Up ---
        # Zwingend für @compile und KeOps Kernel Generierung
        if device == 'cuda': torch.cuda.synchronize()
        if needs_chunk:
            # Kurzer Warmup mit weniger Iterationen
            _ = func(data, r0, c, CHUNK)
        else:
            # Bei Naive/Simple müssen wir aufpassen, dass Warmup nicht crasht
            # Da N hier moderat ist (4096), machen wir Full Run
            _ = func(data, r0, c)
        if device == 'cuda': torch.cuda.synchronize()

        # --- 2. Benchmark ---
        start_time = time.perf_counter()

        if needs_chunk:
            f, p = func(data, r0, c, CHUNK)
        else:
            f, p = func(data, r0, c)

        if device == 'cuda': torch.cuda.synchronize()
        duration = time.perf_counter() - start_time

        # --- 3. Result Parsing ---
        if layout == "DxN":
            f0 = f[:, 0] # (3,)
        else:
            f0 = f[0, :] # (3,)

        # Sicherstellen, dass f0 ein Tensor ist (KeOps gibt manchmal Lazy zurück, hier aber reduction -> Tensor)
        f0_str = f"[{f0[0]:.2f}, {f0[1]:.2f}, {f0[2]:.2f}]"

        # Check auf Fehler/NaN
        if torch.isnan(p) or torch.isinf(p):
            p_val = "NaN/Inf"
        else:
            p_val = f"{p:.4e}"

        print(f"{name:<25} | {duration:.5f} s  | {f0_str:<40} | {p_val}")

    except RuntimeError as e:
        err = str(e)
        if "out of memory" in err:
            print(f"{name:<25} | {'OOM (VRAM)':<10} | {'-'*40} | -")
        else:
            print(f"{name:<25} | {'ERROR':<10} | {err[:40]}... | -")
        # Cache leeren
        torch.cuda.empty_cache()

print("-" * 100)
print("Interpretation:")
print("1. Manual Chunked (NxD) sollte 'Manual Chunked (DxN)' schlagen (Speicherzugriff).")
print("2. Torch Compiled Chunked sollte 'Manual Chunked' schlagen (Kernel Fusion).")
print("3. Naive & Simple sind schnell bei kleinen N, crashen aber bei N > 15k.")
print("4. KeOps & Compiled Chunked sind die einzigen, die für riesige N skalieren.")

In [ ]:
import torch
import time
import sys

# Versuche KeOps zu importieren
try:
    from pykeops.torch import LazyTensor
    has_keops = True
except ImportError:
    has_keops = False
    print("INFO: PyKeOps nicht installiert. KeOps Tests werden übersprungen.\n")

# --- KONFIGURATION ---
N = 40        # Teilchenzahl (Safe für alle Methoden)
D = 3           # Dimensionen
CHUNK = 1024    # Blockgröße
r0 = .1
c = 1
dtype = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"===========================================================")
print(f"   ULTIMATIVER N-BODY BENCHMARK (NxD vs DxN)")
print(f"   N={N}, Device={device}, Chunk={CHUNK}")
print(f"===========================================================\n")

# ============================================================================
# 1. GRUPPE: MANUELLE IMPLEMENTIERUNGEN (Loop, Naive, Manual Chunked)
# ============================================================================
@torch.compile(mode="max-autotune")
def naive_DxN(q, r0, c):
    r0_2 = r0**2; factor = c / r0_2
    diff = q.unsqueeze(2) - q.unsqueeze(1) # (3, N, N)
    r_sq = (diff**2).sum(dim=0)
    exp_term = torch.exp(-r_sq / (2 * r0_2))
    forces = (diff * (exp_term * factor).unsqueeze(0)).sum(dim=2)
    pot = 0.5 * ((c * exp_term).sum() - q.shape[1] * c)
    return forces, pot
@torch.compile(mode="max-autotune")
def naive_NxD(q, r0, c):
    r0_2 = r0**2; factor = c / r0_2
    diff = q.unsqueeze(1) - q.unsqueeze(0) # (N, N, 3)
    r_sq = (diff**2).sum(dim=2)
    exp_term = torch.exp(-r_sq / (2 * r0_2))
    forces = (diff * (exp_term * factor).unsqueeze(2)).sum(dim=1)
    pot = 0.5 * ((c * exp_term).sum() - q.shape[0] * c)
    return forces, pot
@torch.compile(mode="max-autotune")
def loop_DxN(q, r0, c):
    D, N = q.shape; r0_2 = r0**2; factor = c / r0_2
    forces = torch.zeros_like(q); pot_acc = 0.0
    for i in range(N):
        q_i = q[:, i:i+1]
        diff = q_i - q
        r_sq = (diff**2).sum(dim=0)
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[:, i] = (diff * (exp_term * factor).unsqueeze(0)).sum(dim=1)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)
@torch.compile(mode="max-autotune")
def loop_NxD(q, r0, c):
    N, D = q.shape; r0_2 = r0**2; factor = c / r0_2
    forces = torch.zeros_like(q); pot_acc = 0.0
    for i in range(N):
        q_i = q[i:i+1]
        diff = q_i - q
        r_sq = (diff**2).sum(dim=1)
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[i] = (diff * (exp_term * factor).unsqueeze(1)).sum(dim=0)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)
@torch.compile(mode="max-autotune")
def manual_chunked_DxN(q, r0, c, chunk):
    D, N = q.shape; r0_2 = r0**2; factor = c / r0_2
    forces = torch.empty_like(q); pot_acc = 0.0
    for i in range(0, N, chunk):
        end = min(i+chunk, N)
        q_chunk = q[:, i:end] # (3, B)
        diff = q_chunk.unsqueeze(2) - q.unsqueeze(1) # (3, B, N)
        r_sq = (diff**2).sum(dim=0) # Stride Problem!
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[:, i:end] = (diff * (exp_term * factor).unsqueeze(0)).sum(dim=2)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)
@torch.compile(mode="max-autotune")
def manual_chunked_NxD(q, r0, c, chunk):
    N, D = q.shape; r0_2 = r0**2; factor = c / r0_2
    forces = torch.empty_like(q); pot_acc = 0.0
    for i in range(0, N, chunk):
        end = min(i+chunk, N)
        q_chunk = q[i:end] # (B, 3)
        diff = q_chunk.unsqueeze(1) - q.unsqueeze(0) # (B, N, 3)
        r_sq = (diff**2).sum(dim=2) # Gut!
        exp_term = torch.exp(-r_sq / (2 * r0_2))
        forces[i:end] = (diff * (exp_term * factor).unsqueeze(2)).sum(dim=1)
        pot_acc += (c * exp_term).sum()
    return forces, 0.5 * (pot_acc - N * c)


# ============================================================================
# 2. GRUPPE: TORCH COMPILE (JIT Optimized)
# ============================================================================

# --- 2.1 Compiled Chunked NxD ---
@torch.compile(mode="max-autotune")
def _step_compiled_NxD(x_i_chunk, x_j_all, r0_2, c):
    # x_i_chunk: (B, 3), x_j_all: (N, 3)
    R_vec = x_i_chunk[:, None, :] - x_j_all[None, :, :] # (B, N, 3)
    r_sq = torch.sum(R_vec**2, dim=2)                   # Sum over (x,y,z) last dim
    exp_term = (-r_sq / (2 * r0_2)).exp()
    pot_sum = torch.sum(c * exp_term, dim=1)
    # Force: (B, N, 3) * (B, N, 1) -> sum(dim=1)
    forces = torch.sum((c * exp_term).unsqueeze(2) * R_vec / r0_2, dim=1)
    return forces, pot_sum
@torch.compile(mode="max-autotune")
def compiled_chunked_NxD(positions, r0, c, chunk_size):
    N = positions.shape[0]; r0_2 = r0**2
    forces_list = []; pot_list = []
    for x_i in positions.split(chunk_size):
        f, p = _step_compiled_NxD(x_i, positions, r0_2, c)
        forces_list.append(f); pot_list.append(p)
    forces = torch.cat(forces_list, dim=0)
    pot = 0.5 * (torch.cat(pot_list, dim=0).sum() - N * c)
    return forces, pot

# --- 2.2 Compiled Chunked DxN (NEU) ---
@torch.compile(mode="max-autotune")
def _step_compiled_DxN(x_i_chunk, x_j_all, r0_2, c):
    # x_i_chunk: (3, B), x_j_all: (3, N)
    # Broadcasting zu (3, B, N)
    R_vec = x_i_chunk.unsqueeze(2) - x_j_all.unsqueeze(1)

    # r^2: Summe über Dimension 0 (die 3 Koordinaten)
    # Das ist der Schritt mit dem schlechten Stride
    r_sq = torch.sum(R_vec**2, dim=0)

    exp_term = (-r_sq / (2 * r0_2)).exp() # (B, N)
    pot_sum = torch.sum(c * exp_term, dim=1)

    # Force: (3, B, N) * (1, B, N) -> sum(dim=2) über alle N
    forces = torch.sum((c * exp_term).unsqueeze(0) * R_vec / r0_2, dim=2)
    return forces, pot_sum
@torch.compile(mode="max-autotune")
def compiled_chunked_DxN(positions, r0, c, chunk_size):
    D, N = positions.shape; r0_2 = r0**2
    forces_list = []; pot_list = []
    # split entlang Dimension 1 (Spalten)
    for x_i in positions.split(chunk_size, dim=1):
        f, p = _step_compiled_DxN(x_i, positions, r0_2, c)
        forces_list.append(f); pot_list.append(p)
    forces = torch.cat(forces_list, dim=1) # Cat entlang Dimension 1
    pot = 0.5 * (torch.cat(pot_list, dim=0).sum() - N * c)
    return forces, pot


# ============================================================================
# 3. GRUPPE: PYKEOPS (Symbolic Map-Reduce)
# ============================================================================

# --- 3.1 KeOps NxD ---
def keops_NxD(positions, r0, c):
    if not has_keops: return None, 0.0
    N = positions.shape[0]; r0_2 = r0**2

    # NxD Input ist nativ für KeOps
    x_i = LazyTensor(positions[:, None, :]) # (N, 1, 3)
    x_j = LazyTensor(positions[None, :, :]) # (1, N, 3)

    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term

    pot = 0.5 * (potential_ij.sum_reduction(axis=1).sum() - N*c)
    forces = (potential_ij * diff / r0_2).sum_reduction(axis=1)
    return forces, pot

# --- 3.2 KeOps DxN (NEU) ---
def keops_DxN(positions, r0, c):
    if not has_keops: return None, 0.0
    D, N = positions.shape; r0_2 = r0**2

    # KeOps erwartet Vektoren in der letzten Dimension.
    # Bei DxN (3, N) müssen wir transponieren, um LazyTensor korrekt zu füttern.
    # Wir nutzen .t() (Transpose View), das ist sehr billig (kein Copy),
    # aber es erzeugt non-contiguous memory für KeOps (Stride Test!).
    pos_T = positions.t() # (N, 3)

    x_i = LazyTensor(pos_T[:, None, :])
    x_j = LazyTensor(pos_T[None, :, :])

    diff = x_i - x_j
    r_sq = diff.sqnorm2()
    exp_term = (-r_sq / (2 * r0_2)).exp()
    potential_ij = c * exp_term

    pot = 0.5 * (potential_ij.sum_reduction(axis=1).sum() - N*c)

    # Forces kommen als (N, 3) raus
    forces_NxD = (potential_ij * diff / r0_2).sum_reduction(axis=1)

    # Wir müssen zurück transponieren auf DxN für den Output
    return forces_NxD.t(), pot


# ============================================================================
# MAIN BENCHMARK RUNNER
# ============================================================================

# 1. Daten initialisieren
q_NxD = torch.randn(N, D, device=device, dtype=dtype).contiguous()
q_DxN = q_NxD.T.contiguous() # Echtes DxN Layout im Speicher

# Liste: (Name, Funktion, InputDaten, OutputLayout, BrauchtChunkArg)
algos = [
    # Manuell
#    ("Naive           (DxN)", naive_DxN,            q_DxN, "DxN", False),
#    ("Naive           (NxD)", naive_NxD,            q_NxD, "NxD", False),
#    ("Loop            (DxN)", loop_DxN,             q_DxN, "DxN", False),
#    ("Loop            (NxD)", loop_NxD,             q_NxD, "NxD", False),
    ("Manual Chunked  (DxN)", manual_chunked_DxN,   q_DxN, "DxN", True),
    ("Manual Chunked  (NxD)", manual_chunked_NxD,   q_NxD, "NxD", True),

    # Compiled
    ("Torch JIT Chunk (DxN)", compiled_chunked_DxN, q_DxN, "DxN", True),
    ("Torch JIT Chunk (NxD)", compiled_chunked_NxD, q_NxD, "NxD", True),
]

if has_keops:
    algos.append(("PyKeOps         (NxD)", keops_NxD, q_NxD, "NxD", False))
    algos.append(("PyKeOps         (DxN)", keops_DxN, q_DxN, "DxN", False))

print(f"{'ALGORITHMUS':<25} | {'ZEIT (s)':<10} | {'KRAFT TEILCHEN 0 (X, Y, Z)':<40} | {'POTENTIAL'}")
print("-" * 100)

for name, func, data, layout, needs_chunk in algos:
    try:
        # --- A. WARM-UP (Kritisch für JIT/KeOps) ---
        if device == 'cuda': torch.cuda.synchronize()
        if needs_chunk:
            # Kurzer Warmup (wir nutzen vollen Chunk damit Compiler optimiert)
            _ = func(data, r0, c, CHUNK)
        else:
            # Bei Naive Full Run (bei 4096 ok)
            _ = func(data, r0, c)
        if device == 'cuda': torch.cuda.synchronize()

        # --- B. MESSUNG ---
        start = time.perf_counter()

        if needs_chunk:
            f, p = func(data, r0, c, CHUNK)
        else:
            f, p = func(data, r0, c)

        if device == 'cuda': torch.cuda.synchronize()
        duration = time.perf_counter() - start

        # --- C. CHECK & PRINT ---
        # Kraft extrahieren
        if layout == "DxN":
            f0 = f[:, 0] # (3,)
        else:
            f0 = f[0, :] # (3,)

        # Formatierung
        f0_str = f"[{f0[0]:.2f}, {f0[1]:.2f}, {f0[2]:.2f}]"

        # Potential Check
        if torch.isnan(p) or torch.isinf(p): p_str = "NaN/Inf"
        else: p_str = f"{p:.4e}"

        print(f"{name:<25} | {duration:.5f} s  | {f0_str:<40} | {p_str}")

    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"{name:<25} | {'OOM (VRAM)':<10} | {'-'*40} | -")
        else:
            print(f"{name:<25} | {'ERROR':<10} | {str(e)[:30]}... | -")
        torch.cuda.empty_cache()

print("-" * 100)